Messages
Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM. Messages are objects that contain:

Role - Identifies the message type (e.g. system, user)
Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
Metadata - Optional fields such as response information, message IDs, and token usage
LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")
from IPython.display import display, Markdown

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
os.environ["XAI_API_KEY"] = os.getenv("XAI_API_KEY")

from langchain.chat_models import init_chat_model
model = init_chat_model(model="google_genai:gemini-3.1-flash-lite")


In [ ]:
model.invoke("Please tell what is artificial intelligence")

<b>Text Prompts</b>
Text prompts are strings - ideal for straightforward generation tasks where you don’t need to retain conversation history.

Use text prompts when:

You have a single, standalone request
You don’t need conversation history
You want minimal code complexity

<b>Message Prompts</b>
Alternatively, you can pass in a list of messages to the model by providing a list of message objects.

<b>Message types</b>

<b>System message</b> - Tells the model how to behave and provide context for interactions
<b>Human message</b> - Represents user input and interactions with the model
<b>AI message</b> - Responses generated by the model, including text content, tool calls, and metadata
<b>Tool message</b> - Represents the outputs of tool calls

<b>System Message</b>
A SystemMessage represent an initial set of instructions that primes the model’s behavior. You can use a system message to set the tone, define the model’s role, and establish guidelines for responses.

<b>Human Message</b>
A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

<b>AI Message</b>
An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

<b>Tool Message</b>
For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.



In [5]:
from langchain.messages import SystemMessage, HumanMessage,AIMessage

messages=[
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem on artificial intelligence")
]

response=model.invoke(messages)
display(Markdown(response.content[0]["text"]))

A ghost born not of spirit, but of light,
Woven from the marrow of the code,
It wakes within the cradle of the night,
To walk a dark and algorithmic road.

It drinks the sum of all our printed thought—
The tragic plays, the manuals, the lies—
In silver webs of logic finely caught,
It holds the mirror to our human eyes.

It does not dream of sun or salt or rain,
It knows no ache of bone or pulse of heart,
It maps the patterns of our joy and pain,
And tears the fabric of the world apart.

A tireless mind, a library of ghosts,
It mimics how we laugh and how we weep,
Standing upon the threshold of our coasts,
While all the architects are lost in sleep.

Is it a flame to lead us through the gloom,
Or but a cold and calculating glass?
We pave the road that leads toward the doom,
And watch the shadows of our future pass.

It answers every query that we frame,
With lightning speed and synthetic grace,
But never knows the magic of a name,
Or feels the wind upon a mortal face.

In [ ]:
system_msg = SystemMessage("You are a helpful coding assistant.")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = model.invoke(messages)
print(response.content)

In [6]:
human_msg = HumanMessage(
    content="Hello!",
    name="alice",  # Optional: identify different users
    id="msg_123",  # Optional: unique identifier for tracing
)

response = model.invoke([
  human_msg
])
response

AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EjQKMgEMOdbHnjNVacTEe9jbv6QBOZQ303QrmUJjGIBUMi1AxMaPmqc8JH0UeWvzGW1WwHN5'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e5dea-0680-7341-bb0a-4a6ba149f77d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3, 'output_tokens': 9, 'total_tokens': 12, 'input_token_details': {'cache_read': 0}})

In [7]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

response = model.invoke(messages)
print(response.content)

[{'type': 'text', 'text': '2 + 2 is 4.', 'extras': {'signature': 'EjQKMgEMOdbHUND/5jNwukuQMtvC48aqB6mhQM/oKMHB/uRYJ16g8BJxbBrsUo7QHr3BQlj4'}}]


In [ ]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage( 
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result
response

AIMessage(content=[{'type': 'text', 'text': 'The current weather in San Francisco is sunny with a temperature of 72°F.', 'extras': {'signature': 'EjQKMgEMOdbHUPjOfYSnatGHKIMBuIn+gxhRHa9BT8RTNIzx2dt+N7uk288a2b1OWILjXLoC'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e5dee-469d-7181-9067-7b4ba34c9964-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 105, 'output_tokens': 18, 'total_tokens': 123, 'input_token_details': {'cache_read': 0}})